# MiniDiLoCo – Optimizer Benchmark

This notebook runs a **grid search** over inner × outer optimizer combinations
using a single GPU (or CPU) and a short training run.  
Results are collected in a DataFrame and visualised as a heatmap.

The benchmark uses the **UResNet** model extracted from `model_benchmark.ipynb`
and the standard Wikitext-2 dataloader from MiniDiLoCo.

> **Tip:** adjust `TOTAL_STEPS` and `H` to trade speed for accuracy.

In [ ]:
import os, json, time, itertools
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy

# ── single-process setup (rank 0, world_size 1) ──────────────────────────────
os.environ.setdefault('MASTER_ADDR', 'localhost')
os.environ.setdefault('MASTER_PORT', '12355')
os.environ.setdefault('LOCAL_RANK', '0')
os.environ.setdefault('WORLD_SIZE', '1')

RANK       = int(os.environ['LOCAL_RANK'])
WORLD_SIZE = int(os.environ['WORLD_SIZE'])

DEVICE  = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
BACKEND = 'nccl' if torch.cuda.is_available() else 'gloo'

from utils import setup, cleanup
setup(BACKEND, RANK, WORLD_SIZE)

In [ ]:
# ── benchmark hyper-parameters ───────────────────────────────────────────────
TOTAL_STEPS   = 5     # outer steps per run  (increase for more reliable results)
H             = 20    # inner steps per outer step
BATCH_SIZE    = 4
SEQ_LENGTH    = 128
WARMUP_STEPS  = 10

In [ ]:
from strategy import make_adamw, make_adam, make_sgd, make_rmsprop

# Optimizer factories to benchmark
INNER_OPTS = {
    'AdamW(4e-4)' : make_adamw(lr=4e-4),
    'AdamW(1e-3)' : make_adamw(lr=1e-3),
    'Adam(4e-4)'  : make_adam(lr=4e-4),
    'RMSprop(1e-3)': make_rmsprop(lr=1e-3),
}

OUTER_OPTS = {
    'SGD-Nesterov(0.7)' : make_sgd(lr=0.7,  momentum=0.9, nesterov=True),
    'SGD-Nesterov(0.1)' : make_sgd(lr=0.1,  momentum=0.9, nesterov=True),
    'SGD(0.7)'          : make_sgd(lr=0.7,  momentum=0.0, nesterov=False),
    'Adam(1e-2)'        : make_adam(lr=1e-2),
}

In [ ]:
from model import get_gpt2_model
from data import get_dataloader
from train import Trainer
from strategy import Diloco

results = []

for (inner_name, inner_factory), (outer_name, outer_factory) in itertools.product(
    INNER_OPTS.items(), OUTER_OPTS.items()
):
    print(f'\n→ inner={inner_name}  outer={outer_name}')

    model      = get_gpt2_model()
    dataloader = get_dataloader(
        rank=RANK, world_size=WORLD_SIZE,
        batch_size=BATCH_SIZE, seq_length=SEQ_LENGTH
    )

    strategy = Diloco(
        inner_optimizer_factory=inner_factory,
        outer_optimizer_factory=outer_factory,
        warmup_steps=WARMUP_STEPS,
        H=H,
    )

    trainer = Trainer(RANK, WORLD_SIZE, model, dataloader)
    t0      = time.time()
    history = trainer.train(strategy, total_steps=TOTAL_STEPS)
    elapsed = time.time() - t0

    results.append({
        'inner_opt'  : inner_name,
        'outer_opt'  : outer_name,
        'final_loss' : history[-1],
        'min_loss'   : min(history),
        'elapsed_s'  : round(elapsed, 1),
    })
    print(f'   final_loss={history[-1]:.4f}  min_loss={min(history):.4f}  ({elapsed:.1f}s)')

df = pd.DataFrame(results)
print('\nBenchmark complete.')
df

In [ ]:
# ── heatmap: final loss per inner × outer combination ────────────────────────
pivot = df.pivot(index='inner_opt', columns='outer_opt', values='final_loss')

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='RdYlGn_r',
    linewidths=0.5, ax=ax
)
ax.set_title('DiLoCo – Final Loss  (inner × outer optimizer)', fontsize=14, pad=12)
ax.set_xlabel('Outer optimizer')
ax.set_ylabel('Inner optimizer')
plt.tight_layout()
plt.savefig('benchmark_heatmap.png', dpi=150)
plt.show()
print('Saved benchmark_heatmap.png')

In [ ]:
# ── bar chart: best combination per inner optimizer ───────────────────────────
best = df.loc[df.groupby('inner_opt')['final_loss'].idxmin()].copy()
best['label'] = best['inner_opt'] + ' + ' + best['outer_opt']
best = best.sort_values('final_loss')

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(best['label'], best['final_loss'], color='steelblue')
ax.set_xlabel('Final loss (lower is better)')
ax.set_title('Best outer optimizer for each inner optimizer')
plt.tight_layout()
plt.savefig('benchmark_best.png', dpi=150)
plt.show()

In [ ]:
df.to_csv('benchmark_results.csv', index=False)
print(df.to_string(index=False))
cleanup()